# Model Training — Employee Attrition

This notebook trains and compares four classification models using the feature-engineered and selected dataset:

1. Logistic Regression
2. Random Forest
3. Support Vector Machine (SVM)
4. Gradient Boosting

Because the target is imbalanced, `class_weight="balanced"` is used where supported and balanced sample weights are used for Gradient Boosting.

In [31]:
import pandas as pd
import numpy as np
import joblib
from pathlib import Path

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.svm import SVC

from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, average_precision_score, confusion_matrix,
    classification_report
)


In [32]:
data_candidates = [
    Path("../data/employee_attrition_selected.csv"),
    Path("backend/data/employee_attrition_selected.csv"),
    Path("/mnt/data/employee_attrition_selected.csv")
]

data_path = next((p for p in data_candidates if p.exists()), None)
if data_path is None:
    raise FileNotFoundError("employee_attrition_selected.csv not found.")

df = pd.read_csv(data_path)
print("Dataset shape:", df.shape)
df.head()

Dataset shape: (1470, 33)


,Age,BusinessTravel,Department,DistanceFromHome,Education,EducationField,EnvironmentSatisfaction,Gender,JobInvolvement,JobLevel,...,YearsAtCompany,YearsInCurrentRole,YearsSinceLastPromotion,YearsWithCurrManager,CompanyExperienceRatio,PromotionFrequency,IncomePerYearExperience,SatisfactionScore,RoleTenureRatio,Attrition
0,41,Travel_Rarely,Sales,1,2,Life Sciences,2,Female,3,2,...,6,4,0,5,0.750000,0.000,749.125000,2.00,0.666667,Yes
1,49,Travel_Frequently,Research & Development,8,1,Life Sciences,3,Male,2,2,...,10,7,1,7,1.000000,0.100,513.000000,3.00,0.700000,No
2,37,Travel_Rarely,Research & Development,2,2,Other,4,Male,2,1,...,0,0,0,0,0.000000,0.000,298.571429,3.00,0.000000,Yes
3,33,Travel_Frequently,Research & Development,3,4,Life Sciences,4,Female,3,1,...,8,7,3,0,1.000000,0.375,363.625000,3.25,0.875000,No
4,27,Travel_Rarely,Research & Development,2,1,Medical,1,Male,3,1,...,2,2,2,2,0.333333,1.000,578.000000,2.50,1.000000,No


In [33]:
X = df.drop("Attrition", axis=1)
y = df["Attrition"].map({"Yes": 1, "No": 0})

print("Number of features:", X.shape[1])
print("Target distribution:")
print(y.value_counts())
print("\nTarget proportions:")
print(y.value_counts(normalize=True).round(4))

Number of features: 32
Target distribution:
Attrition
0    1233
1     237
Name: count, dtype: int64

Target proportions:
Attrition
0    0.8388
1    0.1612
Name: proportion, dtype: float64


In [34]:
categorical_features = X.select_dtypes(include=["object"]).columns.tolist()
numeric_features = X.select_dtypes(exclude=["object"]).columns.tolist()

print("Categorical features:", categorical_features)
print("\nNumerical features:", numeric_features)

Categorical features: ['BusinessTravel', 'Department', 'EducationField', 'Gender', 'JobRole', 'MaritalStatus', 'OverTime']

Numerical features: ['Age', 'DistanceFromHome', 'Education', 'EnvironmentSatisfaction', 'JobInvolvement', 'JobLevel', 'JobSatisfaction', 'MonthlyIncome', 'NumCompaniesWorked', 'PercentSalaryHike', 'PerformanceRating', 'RelationshipSatisfaction', 'StockOptionLevel', 'TotalWorkingYears', 'TrainingTimesLastYear', 'WorkLifeBalance', 'YearsAtCompany', 'YearsInCurrentRole', 'YearsSinceLastPromotion', 'YearsWithCurrManager', 'CompanyExperienceRatio', 'PromotionFrequency', 'IncomePerYearExperience', 'SatisfactionScore', 'RoleTenureRatio']


In [35]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print("Training samples:", len(X_train))
print("Testing samples:", len(X_test))

Training samples: 1176
Testing samples: 294


## Preprocessing

- Numerical features → StandardScaler
- Categorical features → OneHotEncoder
- `handle_unknown="ignore"` prevents unseen test categories from causing errors.

Scaling is important for Logistic Regression and SVM. It does not materially affect tree-based models, but keeping the same pipeline makes the comparison consistent.

In [36]:
preprocessor = ColumnTransformer(
    transformers=[
        ("numeric", StandardScaler(), numeric_features),
        (
            "categorical",
            OneHotEncoder(handle_unknown="ignore", sparse_output=False),
            categorical_features
        )
    ]
)

## Model 1 — Logistic Regression

In [37]:
logistic_model = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        (
            "model",
            LogisticRegression(
                max_iter=1000,
                class_weight="balanced",
                random_state=42
            )
        )
    ]
)

logistic_model.fit(X_train, y_train)
print("Logistic Regression trained successfully!")

Logistic Regression trained successfully!


## Model 2 — Random Forest

In [38]:
random_forest_model = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        (
            "model",
            RandomForestClassifier(
                n_estimators=300,
                class_weight="balanced",
                random_state=42
            )
        )
    ]
)

random_forest_model.fit(X_train, y_train)
print("Random Forest trained successfully!")

Random Forest trained successfully!


## Model 3 — Support Vector Machine (SVM)

In [39]:
svm_model = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        (
            "model",
            SVC(
                kernel="rbf",
                class_weight="balanced",
                probability=True,
                random_state=42
            )
        )
    ]
)

svm_model.fit(X_train, y_train)
print("SVM trained successfully!")

SVM trained successfully!


## Model 4 — Gradient Boosting

In [40]:
# GradientBoostingClassifier does not expose class_weight,
# so balanced sample weights are supplied during fitting.
class_counts = y_train.value_counts()
sample_weights = np.where(
    y_train == 1,
    len(y_train) / (2 * class_counts[1]),
    len(y_train) / (2 * class_counts[0])
)

gradient_boosting_model = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        (
            "model",
            GradientBoostingClassifier(random_state=42)
        )
    ]
)

gradient_boosting_model.fit(
    X_train, y_train,
    model__sample_weight=sample_weights
)
print("Gradient Boosting trained successfully!")

Gradient Boosting trained successfully!


## Evaluate all models

In [41]:
models = {
    "Logistic Regression": logistic_model,
    "Random Forest": random_forest_model,
    "SVM": svm_model,
    "Gradient Boosting": gradient_boosting_model
}

results = []
predictions = {}
probabilities = {}

for model_name, model in models.items():
    y_pred = model.predict(X_test)
    y_prob = model.predict_proba(X_test)[:, 1]

    predictions[model_name] = y_pred
    probabilities[model_name] = y_prob

    results.append({
        "Model": model_name,
        "Accuracy": accuracy_score(y_test, y_pred),
        "Precision": precision_score(y_test, y_pred, zero_division=0),
        "Recall": recall_score(y_test, y_pred, zero_division=0),
        "F1 Score": f1_score(y_test, y_pred, zero_division=0),
        "ROC-AUC": roc_auc_score(y_test, y_prob),
        "PR-AUC": average_precision_score(y_test, y_prob)
    })

results_df = pd.DataFrame(results).sort_values(
    by=["F1 Score", "Recall", "ROC-AUC"],
    ascending=False
).reset_index(drop=True)

results_df.round(4)

,Model,Accuracy,Precision,Recall,F1 Score,ROC-AUC,PR-AUC
0,SVM,0.8197,0.4516,0.5957,0.5138,0.8105,0.5368
1,Logistic Regression,0.7721,0.3864,0.7234,0.5037,0.8188,0.5535
2,Gradient Boosting,0.7959,0.3818,0.4468,0.4118,0.7774,0.4670
3,Random Forest,0.8435,0.5556,0.1064,0.1786,0.8105,0.4442


## Model Selection

Since this is an imbalanced binary classification problem and no specific business cost function has been defined, **F1-score is used as the primary model-selection metric** because it provides a balance between Precision and Recall. **Recall, Precision, ROC-AUC, and PR-AUC are used as supporting metrics**, while Accuracy is reported but is not used alone to select the final model.

The model with the highest F1-score is selected as the final model. If two models have the same F1-score, Recall and then ROC-AUC are used as tie-breakers.

In [42]:
best_model_name = results_df.iloc[0]["Model"]
best_model = models[best_model_name]

print("Model ranking (F1-score first):")
for i, row in results_df.iterrows():
    print(
        f"{i+1}. {row['Model']} | "
        f"F1={row['F1 Score']:.4f}, "
        f"Precision={row['Precision']:.4f}, "
        f"Recall={row['Recall']:.4f}, "
        f"ROC-AUC={row['ROC-AUC']:.4f}, "
        f"PR-AUC={row['PR-AUC']:.4f}"
    )

print("\nSelected final model:", best_model_name)

Model ranking (F1-score first):
1. SVM | F1=0.5138, Precision=0.4516, Recall=0.5957, ROC-AUC=0.8105, PR-AUC=0.5368
2. Logistic Regression | F1=0.5037, Precision=0.3864, Recall=0.7234, ROC-AUC=0.8188, PR-AUC=0.5535
3. Gradient Boosting | F1=0.4118, Precision=0.3818, Recall=0.4468, ROC-AUC=0.7774, PR-AUC=0.4670
4. Random Forest | F1=0.1786, Precision=0.5556, Recall=0.1064, ROC-AUC=0.8105, PR-AUC=0.4442

Selected final model: SVM


## Confusion matrices and classification reports

In [43]:
for model_name in models:
    print("\n" + "=" * 55)
    print(model_name)
    print("=" * 55)
    print("Confusion Matrix:")
    print(confusion_matrix(y_test, predictions[model_name]))
    print("\nClassification Report:")
    print(classification_report(
        y_test, predictions[model_name],
        target_names=["No Attrition", "Attrition"],
        zero_division=0
    ))


Logistic Regression
Confusion Matrix:
[[193  54]
 [ 13  34]]

Classification Report:
              precision    recall  f1-score   support

No Attrition       0.94      0.78      0.85       247
   Attrition       0.39      0.72      0.50        47

    accuracy                           0.77       294
   macro avg       0.66      0.75      0.68       294
weighted avg       0.85      0.77      0.80       294


Random Forest
Confusion Matrix:
[[243   4]
 [ 42   5]]

Classification Report:
              precision    recall  f1-score   support

No Attrition       0.85      0.98      0.91       247
   Attrition       0.56      0.11      0.18        47

    accuracy                           0.84       294
   macro avg       0.70      0.55      0.55       294
weighted avg       0.81      0.84      0.80       294


SVM
Confusion Matrix:
[[213  34]
 [ 19  28]]

Classification Report:
              precision    recall  f1-score   support

No Attrition       0.92      0.86      0.89       247
 

## Save the final model and comparison results

In [44]:
model_path = Path("../models/attrition_model.pkl")
model_path.parent.mkdir(parents=True, exist_ok=True)
joblib.dump(best_model, model_path)

results_path = model_path.parent / "model_comparison.csv"
results_df.to_csv(results_path, index=False)

print(f"Final model ({best_model_name}) saved successfully!")
print("Model path:", model_path)
print("Comparison saved at:", results_path)

Final model (SVM) saved successfully!
Model path: ..\models\attrition_model.pkl
Comparison saved at: ..\models\model_comparison.csv


In [45]:
import json

metrics = {
    "selected_model": best_model_name,
    "selection_priority": ["F1 Score", "Recall", "ROC-AUC"],
    "models": {
        row["Model"]: {
            "accuracy": round(row["Accuracy"], 4),
            "precision": round(row["Precision"], 4),
            "recall": round(row["Recall"], 4),
            "f1_score": round(row["F1 Score"], 4),
            "roc_auc": round(row["ROC-AUC"], 4),
            "pr_auc": round(row["PR-AUC"], 4)
        }
        for _, row in results_df.iterrows()
    }
}

metrics_path = model_path.parent / "metrics.json"
with open(metrics_path, "w") as f:
    json.dump(metrics, f, indent=4)

print("Metrics saved at:", metrics_path)

Metrics saved at: ..\models\metrics.json
